# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [34]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [35]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


**Check the shape of your data**

In [36]:
spaceship.shape #8693 rows and 14 columns

(8693, 14)

**Check for data types**

In [37]:
spaceship.dtypes

PassengerId      object
HomePlanet       object
CryoSleep        object
Cabin            object
Destination      object
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name             object
Transported        bool
dtype: object

**Check for missing values**

In [38]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [39]:
spaceship[spaceship.isnull().any(axis=1)] # there are 2087 rows out of 8693 that has some missing values, so it is 25% of the data

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,True
10,0008_02,Europa,True,B/1/P,TRAPPIST-1e,34.0,False,0.0,0.0,NaN,0.0,0.0,Altardr Flatic,True
15,0012_01,Earth,False,NaN,TRAPPIST-1e,31.0,False,32.0,0.0,876.0,0.0,0.0,Justie Pooles,False
16,0014_01,Mars,False,F/3/P,55 Cancri e,27.0,False,1286.0,122.0,NaN,0.0,0.0,Flats Eccle,False
23,0020_03,Earth,True,E/0/S,55 Cancri e,29.0,False,0.0,0.0,NaN,0.0,0.0,Mollen Mcfaddennon,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8667,9250_01,Europa,False,E/597/P,TRAPPIST-1e,29.0,False,0.0,2972.0,NaN,28.0,188.0,Chain Reedectied,True
8674,9257_01,NaN,False,F/1892/P,TRAPPIST-1e,13.0,False,39.0,0.0,1085.0,24.0,0.0,Ties Apple,False
8675,9259_01,Earth,NaN,F/1893/P,TRAPPIST-1e,44.0,False,1030.0,1015.0,0.0,11.0,NaN,Annah Gilleyons,True
8684,9274_01,NaN,True,G/1508/P,TRAPPIST-1e,23.0,False,0.0,0.0,0.0,0.0,0.0,Chelsa Bullisey,True


In [40]:
spaceship_cleaned = spaceship.dropna()


In [41]:
spaceship_cleaned.shape

(6606, 14)

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [42]:
spaceship_cleaned['Cabin'] = spaceship_cleaned['Cabin'].str[0]

C:\Users\Anna\AppData\Local\Temp\ipykernel_19160\2530471862.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spaceship_cleaned['Cabin'] = spaceship_cleaned['Cabin'].str[0]


In [43]:
spaceship_cleaned.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


- Drop PassengerId and Name

In [44]:
spaceship_final = spaceship_cleaned.drop(['PassengerId', 'Name'], axis=1)

In [45]:
spaceship_final.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


##### X - y split

In [46]:
X = spaceship_final.drop(columns=['Transported'])
y = spaceship_final['Transported']

In [47]:
X.shape

(6606, 11)

**Perform Train Test Split**

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [49]:
X_train.shape

(5284, 11)

- For non-numerical columns, do dummies.

In [50]:
#separating nominal columns from numerical

nominal_cols = ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']

X_train_nominal = X_train[nominal_cols]
X_train_numerical = X_train.drop(columns=nominal_cols)

X_test_nominal = X_test[nominal_cols]
X_test_numerical = X_test.drop(columns=nominal_cols)

In [51]:
X_train_nominal.shape

(5284, 5)

In [52]:
from sklearn.preprocessing import OneHotEncoder

In [53]:
# encoding nominal features

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

ohe.fit(X_train_nominal) #fitting only on train data to avoid data leakage

X_train_nominal_encoded = ohe.transform(X_train_nominal)
X_test_nominal_encoded = ohe.transform(X_test_nominal)


In [54]:
X_train_nominal_encoded.shape

(5284, 17)

In [55]:
# let's get the column names for encoded features

encoded_cols = ohe.get_feature_names_out()

X_train_nominal_df = pd.DataFrame(X_train_nominal_encoded, columns=encoded_cols, index=X_train.index)
X_test_nominal_df = pd.DataFrame(X_test_nominal_encoded, columns=encoded_cols, index=X_test.index)

In [56]:
# Let's concatenate to create one dataframe for modeling

X_train_final = pd.concat([X_train_nominal_df, X_train_numerical], axis=1)
X_test_final = pd.concat([X_test_nominal_df, X_test_numerical], axis=1)

X_train = X_train_final
X_test = X_test_final

In [57]:
X_train.head()

,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_False,CryoSleep_True,Cabin_A,Cabin_B,Cabin_C,Cabin_D,Cabin_E,...,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
8496,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,27.0,0.0,0.0,0.0,0.0,0.0
2347,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,48.0,639.0,412.0,110.0,0.0,0.0
1654,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,26.0,0.0,0.0,0.0,0.0,0.0
944,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,21.0,0.0,0.0,0.0,0.0,791.0
593,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [58]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

Normalize the data

In [59]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [60]:
normalizer = MinMaxScaler()

normalizer.fit(X_train)

MinMaxScaler()

In [61]:
X_train_norm = normalizer.transform(X_train)

X_test_norm = normalizer.transform(X_test)

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [62]:
bagging_clf = BaggingClassifier(DecisionTreeClassifier(max_depth=20),
                                n_estimators=100,
                                max_samples=1000)

In [63]:
bagging_clf.fit(X_train_norm, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=20),
                  max_samples=1000, n_estimators=100)

- Evaluate your model

In [64]:
pred = bagging_clf.predict(X_test_norm)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.7965204236006052
              precision    recall  f1-score   support

       False       0.76      0.83      0.80       634
        True       0.83      0.76      0.80       688

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [67]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_samples': [500, 1000, 2000],
    'estimator__max_depth': [5, 10, 20],
    'estimator__min_samples_split': [2, 5, 10]
}

- Run Grid Search

In [68]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=BaggingClassifier(DecisionTreeClassifier()),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train_norm, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters: {'estimator__max_depth': 5, 'estimator__min_samples_split': 5, 'max_samples': 2000, 'n_estimators': 50}
Best CV accuracy: 0.7959854576990338


- Evaluate your model

In [70]:
best_model = grid_search.best_estimator_
pred_best = best_model.predict(X_test_norm)

print("=== Tuned BaggingClassifier ===")
print("Accuracy:", accuracy_score(y_test, pred_best))
print(classification_report(y_test, pred_best))

print("\n=== Final Comparison ===")
print("Baseline BaggingClassifier: 0.7965")
print("Tuned BaggingClassifier:   ", accuracy_score(y_test, pred_best))

=== Tuned BaggingClassifier ===
Accuracy: 0.7965204236006052
              precision    recall  f1-score   support

       False       0.78      0.80      0.79       634
        True       0.81      0.80      0.80       688

    accuracy                           0.80      1322
   macro avg       0.80      0.80      0.80      1322
weighted avg       0.80      0.80      0.80      1322


=== Final Comparison ===
Baseline BaggingClassifier: 0.7965
Tuned BaggingClassifier:    0.7965204236006052
